# Week 3 - Cross-Encoder Reranker Fine-tuning and Evaluation

This week trains and evaluates a Cross-Encoder reranker on retrieval candidates.


# Day 1 - KorQuAD Dataset Analysis and Positive Context Preparation

Prepare KorQuAD question-context pairs for reranker training.


In [1]:
from pathlib import Path
import json
import pandas as pd

input_root = Path("/kaggle/input")

checkpoint_path = next(
    input_root.rglob("week2_dense_retriever.pt")
)

config_path = next(
    input_root.rglob("retrieval_config.json")
)

candidate_path = next(
    input_root.rglob(
        "week2_day7_query_context_candidates.csv"
    )
)

print("Checkpoint:", checkpoint_path)
print("Config:", config_path)
print("Candidates:", candidate_path)

Checkpoint: /kaggle/input/notebooks/dongmyungpark/sentence-embedding-retrieval/week2_retriever_artifacts/week2_dense_retriever.pt
Config: /kaggle/input/notebooks/dongmyungpark/sentence-embedding-retrieval/week2_retriever_artifacts/retrieval_config.json
Candidates: /kaggle/input/notebooks/dongmyungpark/sentence-embedding-retrieval/week2_day7_query_context_candidates.csv


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import (
    AutoTokenizer,
    AutoModel
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


def mean_pooling(
    model_output,
    attention_mask
):
    token_embeddings = (
        model_output.last_hidden_state
    )

    input_mask_expanded = (
        attention_mask
        .unsqueeze(-1)
        .expand(token_embeddings.size())
        .float()
    )

    sum_embeddings = torch.sum(
        token_embeddings
        * input_mask_expanded,
        dim=1
    )

    sum_mask = torch.clamp(
        input_mask_expanded.sum(dim=1),
        min=1e-9
    )

    return sum_embeddings / sum_mask


class RobertaSTSModel(nn.Module):

    def __init__(self, model_name):
        super().__init__()

        self.roberta = AutoModel.from_pretrained(
            model_name
        )

    def encode(
        self,
        input_ids,
        attention_mask
    ):
        model_output = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        return mean_pooling(
            model_output,
            attention_mask
        )


checkpoint = torch.load(
    checkpoint_path,
    map_location=device
)

week2_retriever_model = RobertaSTSModel(
    checkpoint["model_name"]
).to(device)

week2_retriever_model.load_state_dict(
    checkpoint["model_state_dict"]
)

week2_retriever_model.eval()

print("Week 2 retriever loaded")
print("Model:", checkpoint["model_name"])
print(
    "Embedding dimension:",
    checkpoint["embedding_dimension"]
)
print("Device:", device)

config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: klue/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Week 2 retriever loaded
Model: klue/roberta-base
Embedding dimension: 768
Device: cuda


In [3]:
tokenizer_path = next(
    checkpoint_path.parent.rglob("tokenizer")
)

week2_tokenizer = AutoTokenizer.from_pretrained(
    tokenizer_path
)

print("Tokenizer loaded")
print("Tokenizer path:", tokenizer_path)

Tokenizer loaded
Tokenizer path: /kaggle/input/notebooks/dongmyungpark/sentence-embedding-retrieval/week2_retriever_artifacts/tokenizer


In [4]:
import numpy as np

retrieval_data_matches = list(
    input_root.rglob("retrieval_data.json")
)

if len(retrieval_data_matches) > 0:
    retrieval_data_path = retrieval_data_matches[0]

    with open(
        retrieval_data_path,
        "r",
        encoding="utf-8"
    ) as file:
        retrieval_data = json.load(file)

    week2_queries = retrieval_data[
        "positive_queries"
    ]

    week2_corpus = retrieval_data[
        "corpus"
    ]

    week2_gold_indices = np.array(
        retrieval_data[
            "positive_gold_indices"
        ],
        dtype=np.int64
    )

    print("Original retrieval data loaded")
    print("Number of queries:", len(week2_queries))
    print("Corpus size:", len(week2_corpus))
else:
    print(
        "retrieval_data.json was not found."
    )
    print(
        "The candidate CSV is available, "
        "but the full corpus cannot be reconstructed "
        "from the candidate file alone."
    )

Original retrieval data loaded
Number of queries: 220
Corpus size: 519


In [5]:
class SentenceEncoder:

    def __init__(
        self,
        model,
        tokenizer,
        device,
        max_length=64,
        batch_size=32
    ):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.max_length = max_length
        self.batch_size = batch_size

    def encode(self, sentences):
        self.model.eval()
        all_embeddings = []

        with torch.no_grad():
            for start_idx in range(
                0,
                len(sentences),
                self.batch_size
            ):
                batch_sentences = sentences[
                    start_idx:start_idx + self.batch_size
                ]

                encoded = self.tokenizer(
                    batch_sentences,
                    padding=True,
                    truncation=True,
                    max_length=self.max_length,
                    return_tensors="pt"
                )

                encoded = {
                    key: value.to(self.device)
                    for key, value in encoded.items()
                }

                embeddings = self.model.encode(
                    encoded["input_ids"],
                    encoded["attention_mask"]
                )

                embeddings = F.normalize(
                    embeddings,
                    p=2,
                    dim=1
                )

                all_embeddings.append(
                    embeddings.cpu().numpy()
                )

        return np.concatenate(
            all_embeddings,
            axis=0
        ).astype("float32")


week2_encoder = SentenceEncoder(
    model=week2_retriever_model,
    tokenizer=week2_tokenizer,
    device=device,
    max_length=64,
    batch_size=32
)

test_embeddings = week2_encoder.encode(
    week2_queries[:2]
)

print("Checkpoint inference test completed")
print("Test embedding shape:", test_embeddings.shape)

Checkpoint inference test completed
Test embedding shape: (2, 768)


In [6]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 82.5 MB/s eta 0:00:00


In [7]:
import faiss


query_embeddings = week2_encoder.encode(
    week2_queries
)

corpus_embeddings = week2_encoder.encode(
    week2_corpus
)

retrieval_index = faiss.IndexFlatIP(
    corpus_embeddings.shape[1]
)

retrieval_index.add(
    corpus_embeddings
)

retrieved_similarities, retrieved_indices = (
    retrieval_index.search(
        query_embeddings,
        k=10
    )
)

def calculate_recall_at_k(
    retrieved_indices,
    gold_indices,
    k
):
    hits = []

    for retrieved, gold in zip(
        retrieved_indices,
        gold_indices
    ):
        hits.append(
            int(gold in retrieved[:k])
        )

    return np.mean(hits)


def calculate_mrr_at_k(
    retrieved_indices,
    gold_indices,
    k
):
    reciprocal_ranks = []

    for retrieved, gold in zip(
        retrieved_indices,
        gold_indices
    ):
        top_k_retrieved = retrieved[:k]
        matches = np.where(
            top_k_retrieved == gold
        )[0]

        if len(matches) > 0:
            rank = matches[0] + 1
            reciprocal_ranks.append(
                1.0 / rank
            )
        else:
            reciprocal_ranks.append(0.0)

    return np.mean(reciprocal_ranks)


print("Week 2 Retriever Reproduction")
print(
    "Recall@1:",
    f"{calculate_recall_at_k(retrieved_indices, week2_gold_indices, 1):.4f}"
)
print(
    "Recall@5:",
    f"{calculate_recall_at_k(retrieved_indices, week2_gold_indices, 5):.4f}"
)
print(
    "Recall@10:",
    f"{calculate_recall_at_k(retrieved_indices, week2_gold_indices, 10):.4f}"
)
print(
    "MRR@10:",
    f"{calculate_mrr_at_k(retrieved_indices, week2_gold_indices, 10):.4f}"
)

Week 2 Retriever Reproduction
Recall@1: 0.9500
Recall@5: 0.9955
Recall@10: 0.9955
MRR@10: 0.9720


In [8]:
from datasets import load_dataset

korquad = load_dataset(
    "KorQuAD/squad_kor_v1"
)

print("KorQuAD dataset loaded")
print("Available splits:", korquad.keys())

README.md: 0.00B [00:00, ?B/s]

squad_kor_v1/train-00000-of-00001.parque(…):   0%|          | 0.00/11.6M [00:00<?, ?B/s]

squad_kor_v1/validation-00000-of-00001.p(…):   0%|          | 0.00/1.16M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60407 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5774 [00:00<?, ? examples/s]

KorQuAD dataset loaded
Available splits: dict_keys(['train', 'validation'])


In [9]:
reranker_tokenizer = week2_tokenizer

for split in ["train", "validation"]:
    split_data = korquad[split]

    token_lengths = []

    for example in split_data:
        encoded = reranker_tokenizer(
            example["question"],
            example["context"],
            add_special_tokens=True,
            truncation=False
        )

        token_lengths.append(
            len(encoded["input_ids"])
        )

    token_lengths = np.array(token_lengths)

    print(f"[{split}]")
    print("Average token length:", f"{token_lengths.mean():.2f}")
    print("Median token length:", f"{np.median(token_lengths):.2f}")
    print("90th percentile:", f"{np.percentile(token_lengths, 90):.2f}")
    print("95th percentile:", f"{np.percentile(token_lengths, 95):.2f}")
    print("99th percentile:", f"{np.percentile(token_lengths, 99):.2f}")
    print("Maximum token length:", token_lengths.max())

    for max_length in [128, 256, 512]:
        exceeded = (
            token_lengths > max_length
        ).mean() * 100

        print(
            f"Exceeding max_length={max_length}: "
            f"{exceeded:.2f}%"
        )

Token indices sequence length is longer than the specified maximum sequence length for this model (538 > 512). Running this sequence through the model will result in indexing errors


[train]
Average token length: 294.29
Median token length: 265.00
90th percentile: 407.00
95th percentile: 479.00
99th percentile: 659.00
Maximum token length: 5890
Exceeding max_length=128: 100.00%
Exceeding max_length=256: 55.40%
Exceeding max_length=512: 3.68%
[validation]
Average token length: 303.04
Median token length: 261.00
90th percentile: 418.00
95th percentile: 508.35
99th percentile: 985.81
Maximum token length: 1460
Exceeding max_length=128: 100.00%
Exceeding max_length=256: 53.27%
Exceeding max_length=512: 4.87%


In [10]:
for split in ["train", "validation"]:
    split_data = korquad[split]
    truncated_lengths = []

    for example in split_data:
        encoded = reranker_tokenizer(
            example["question"],
            example["context"],
            truncation=True,
            max_length=512
        )

        truncated_lengths.append(
            len(encoded["input_ids"])
        )

    print(f"[{split}]")
    print(
        "Maximum truncated length:",
        max(truncated_lengths)
    )
    print(
        "Average truncated length:",
        f"{np.mean(truncated_lengths):.2f}"
    )

[train]
Maximum truncated length: 512
Average truncated length: 288.90
[validation]
Maximum truncated length: 512
Average truncated length: 290.15


In [11]:
def is_answer_preserved(
    example,
    tokenizer,
    max_length=512
):
    answer_text = example["answers"]["text"][0]
    answer_start = example["answers"]["answer_start"][0]
    answer_end = answer_start + len(answer_text)

    encoded = tokenizer(
        example["question"],
        example["context"],
        truncation="only_second",
        max_length=max_length,
        return_offsets_mapping=True
    )

    offsets = encoded["offset_mapping"]
    sequence_ids = encoded.sequence_ids()

    context_token_offsets = [
        offset
        for offset, sequence_id in zip(
            offsets,
            sequence_ids
        )
        if sequence_id == 1
    ]

    if len(context_token_offsets) == 0:
        return False

    visible_context_end = max(
        end
        for start, end in context_token_offsets
    )

    return answer_end <= visible_context_end


for max_length in [256, 512]:
    for split in ["train", "validation"]:
        split_data = korquad[split]

        preserved_count = sum(
            is_answer_preserved(
                example,
                reranker_tokenizer,
                max_length=max_length
            )
            for example in split_data
        )

        preservation_rate = (
            preserved_count / len(split_data)
        )

        print(
            f"{split} | "
            f"max_length={max_length} | "
            f"answer preservation="
            f"{preservation_rate:.4f}"
        )

train | max_length=256 | answer preservation=0.9033
validation | max_length=256 | answer preservation=0.9032
train | max_length=512 | answer preservation=0.9956
validation | max_length=512 | answer preservation=0.9901


In [12]:
from datasets import Dataset

def build_positive_pairs(split_data):
    positive_pairs = []

    for example in split_data:
        positive_pairs.append(
            {
                "id": example["id"],
                "title": example["title"],
                "question": example["question"],
                "context": example["context"],
                "answer_text": example["answers"]["text"][0],
                "answer_start": example["answers"]["answer_start"][0],
                "label": 1
            }
        )

    return Dataset.from_list(
        positive_pairs
    )


positive_train_dataset = build_positive_pairs(
    korquad["train"]
)

positive_validation_dataset = build_positive_pairs(
    korquad["validation"]
)

print("Positive datasets created")
print(
    "Train examples:",
    len(positive_train_dataset)
)
print(
    "Validation examples:",
    len(positive_validation_dataset)
)
print(
    "Features:",
    positive_train_dataset.column_names
)

Positive datasets created
Train examples: 60407
Validation examples: 5774
Features: ['id', 'title', 'question', 'context', 'answer_text', 'answer_start', 'label']


In [13]:
def validate_answer_alignment(dataset):
    aligned_count = 0

    for example in dataset:
        answer_text = example["answer_text"]
        context = example["context"]

        if answer_text in context:
            aligned_count += 1

    return aligned_count


train_aligned = validate_answer_alignment(
    positive_train_dataset
)

validation_aligned = validate_answer_alignment(
    positive_validation_dataset
)

print("Answer Alignment Validation")
print("---------------------------")
print(
    "Train aligned:",
    train_aligned,
    "/",
    len(positive_train_dataset)
)
print(
    "Validation aligned:",
    validation_aligned,
    "/",
    len(positive_validation_dataset)
)

Answer Alignment Validation
---------------------------
Train aligned: 60407 / 60407
Validation aligned: 5774 / 5774


# Day 1 Experiment Log

Prepared validated KorQuAD positive pairs with answer-aligned contexts.


# Day 2 - Random Negative and Hard Negative Mining

Create random and hard-negative training examples.


In [14]:
# 학습용 문맥 준비
from collections import OrderedDict
import random

SEED = 378
random.seed(SEED)
np.random.seed(SEED)

train_contexts = [
    example["context"]
    for example in korquad["train"]
]

unique_contexts = list(
    OrderedDict.fromkeys(train_contexts)
)

context_to_id = {
    context: index
    for index, context in enumerate(unique_contexts)
}

print("Negative Sampling Setup")
print("Original context count:", len(train_contexts))
print("Unique context count:", len(unique_contexts))
print(
    "Duplicate contexts:",
    len(train_contexts) - len(unique_contexts)
)

Negative Sampling Setup
Original context count: 60407
Unique context count: 9606
Duplicate contexts: 50801


In [15]:
positive_train_records = []

for example in positive_train_dataset:
    context_id = context_to_id[
        example["context"]
    ]

    positive_train_records.append(
        {
            "id": example["id"],
            "title": example["title"],
            "question": example["question"],
            "positive_context": example["context"],
            "positive_context_id": context_id,
            "answer_text": example["answer_text"],
            "answer_start": example["answer_start"],
            "label": 1
        }
    )

positive_train_df = pd.DataFrame(
    positive_train_records
)

print("Positive training records prepared")
print("Number of records:", len(positive_train_df))
print(
    "Unique positive contexts:",
    positive_train_df["positive_context_id"].nunique()
)

Positive training records prepared
Number of records: 60407
Unique positive contexts: 9606


In [16]:
random_negative_records = []

for _, row in positive_train_df.iterrows():
    positive_context_id = row[
        "positive_context_id"
    ]

    negative_context_id = random.randrange(
        len(unique_contexts)
    )

    while (
        negative_context_id
        == positive_context_id
    ):
        negative_context_id = random.randrange(
            len(unique_contexts)
        )

    random_negative_records.append(
        {
            "id": row["id"],
            "title": row["title"],
            "question": row["question"],
            "context": unique_contexts[
                negative_context_id
            ],
            "context_id": negative_context_id,
            "answer_text": "",
            "answer_start": -1,
            "label": 0,
            "negative_type": "random"
        }
    )

random_negative_df = pd.DataFrame(
    random_negative_records
)

print("Random negative dataset created")
print(
    "Number of random negatives:",
    len(random_negative_df)
)
print(
    "Negative label distribution:",
    random_negative_df["label"].value_counts().to_dict()
)

Random negative dataset created
Number of random negatives: 60407
Negative label distribution: {0: 60407}


In [17]:
HARD_NEGATIVE_SEARCH_K = 20

print("Encoding unique contexts...")

unique_context_embeddings = week2_encoder.encode(
    unique_contexts
)

print(
    "Context embedding shape:",
    unique_context_embeddings.shape
)

hard_negative_index = faiss.IndexFlatIP(
    unique_context_embeddings.shape[1]
)

hard_negative_index.add(
    unique_context_embeddings
)

print(
    "Indexed context vectors:",
    hard_negative_index.ntotal
)

Encoding unique contexts...
Context embedding shape: (9606, 768)
Indexed context vectors: 9606


In [18]:
# 하드 네거티브 생성
print("Encoding training questions...")

question_embeddings = week2_encoder.encode(
    positive_train_df["question"].tolist()
)

print(
    "Question embedding shape:",
    question_embeddings.shape
)

hard_similarities, hard_indices = (
    hard_negative_index.search(
        question_embeddings,
        HARD_NEGATIVE_SEARCH_K
    )
)

hard_negative_records = []

for row_idx, row in positive_train_df.iterrows():
    positive_context_id = row[
        "positive_context_id"
    ]

    selected_position = None

    for position, context_id in enumerate(
        hard_indices[row_idx]
    ):
        if context_id != positive_context_id:
            selected_position = position
            break

    if selected_position is None:
        continue

    hard_context_id = int(
        hard_indices[row_idx][selected_position]
    )

    hard_negative_records.append(
        {
            "id": row["id"],
            "title": row["title"],
            "question": row["question"],
            "context": unique_contexts[
                hard_context_id
            ],
            "context_id": hard_context_id,
            "positive_context_id": positive_context_id,
            "similarity": float(
                hard_similarities[
                    row_idx
                ][selected_position]
            ),
            "answer_text": "",
            "answer_start": -1,
            "label": 0,
            "negative_type": "hard"
        }
    )

hard_negative_df = pd.DataFrame(
    hard_negative_records
)

print("Hard negative dataset created")
print(
    "Number of hard negatives:",
    len(hard_negative_df)
)
print(
    "Average similarity:",
    f"{hard_negative_df['similarity'].mean():.4f}"
)
print(
    "Minimum similarity:",
    f"{hard_negative_df['similarity'].min():.4f}"
)
print(
    "Maximum similarity:",
    f"{hard_negative_df['similarity'].max():.4f}"
)

Encoding training questions...
Question embedding shape: (60407, 768)
Hard negative dataset created
Number of hard negatives: 60407
Average similarity: 0.6058
Minimum similarity: 0.3356
Maximum similarity: 0.9055


In [19]:
hard_negative_checked_df = hard_negative_df.merge(
    positive_train_df[
        [
            "id",
            "answer_text",
            "positive_context_id"
        ]
    ],
    on="id",
    how="left",
    suffixes=(
        "_negative",
        "_positive"
    )
)

hard_negative_checked_df[
    "contains_positive_answer"
] = hard_negative_checked_df.apply(
    lambda row:
        row["answer_text_positive"] in row["context"],
    axis=1
)

false_negative_count = (
    hard_negative_checked_df[
        "contains_positive_answer"
    ].sum()
)

valid_hard_negative_df = (
    hard_negative_checked_df[
        ~hard_negative_checked_df[
            "contains_positive_answer"
        ]
    ]
    .copy()
)

print("Hard Negative Validation")
print(
    "Original hard negatives:",
    len(hard_negative_checked_df)
)
print(
    "Potential false negatives:",
    false_negative_count
)
print(
    "Valid hard negatives:",
    len(valid_hard_negative_df)
)

Hard Negative Validation
Original hard negatives: 60407
Potential false negatives: 4241
Valid hard negatives: 56166


In [20]:
context_id_to_titles = {}

for example in korquad["train"]:
    context = example["context"]
    context_id = context_to_id[context]

    if context_id not in context_id_to_titles:
        context_id_to_titles[context_id] = set()

    context_id_to_titles[context_id].add(
        example["title"]
    )

print(
    "Context title mapping created:",
    len(context_id_to_titles)
)

Context title mapping created: 9606


In [21]:
hard_negative_checked_df[
    "negative_titles"
] = hard_negative_checked_df[
    "context_id"
].apply(
    lambda context_id:
        list(context_id_to_titles.get(
            context_id,
            set()
        ))
)

hard_negative_checked_df[
    "same_title_as_positive"
] = hard_negative_checked_df.apply(
    lambda row:
        row["title"] in row["negative_titles"],
    axis=1
)

same_title_count = (
    hard_negative_checked_df[
        "same_title_as_positive"
    ].sum()
)

print("Same-title hard negatives:", same_title_count)

Same-title hard negatives: 15307


In [22]:
valid_hard_negative_df = (
    hard_negative_checked_df[
        (~hard_negative_checked_df[
            "contains_positive_answer"
        ])
        &
        (~hard_negative_checked_df[
            "same_title_as_positive"
        ])
        &
        (hard_negative_checked_df[
            "similarity"
        ] >= 0.70)
    ]
    .copy()
    .reset_index(drop=True)
)

print("Validated Hard Negative Dataset")
print(
    "Number of valid hard negatives:",
    len(valid_hard_negative_df)
)
print(
    "Average similarity:",
    f"{valid_hard_negative_df['similarity'].mean():.4f}"
)
print(
    "Maximum similarity:",
    f"{valid_hard_negative_df['similarity'].max():.4f}"
)

Validated Hard Negative Dataset
Number of valid hard negatives: 2805
Average similarity: 0.7297
Maximum similarity: 0.8599


In [23]:
hard_negative_training_df = (
    valid_hard_negative_df[
        [
            "id",
            "title",
            "question",
            "context",
            "context_id",
            "similarity",
            "label",
            "negative_type"
        ]
    ]
    .copy()
)

hard_negative_training_df["answer_text"] = ""
hard_negative_training_df["answer_start"] = -1

hard_negative_positive_df = (
    positive_train_df
    .sample(
        n=len(hard_negative_training_df),
        random_state=SEED
    )
    .copy()
)

hard_negative_positive_df = (
    hard_negative_positive_df[
        [
            "id",
            "title",
            "question",
            "positive_context",
            "positive_context_id",
            "answer_text",
            "answer_start",
            "label"
        ]
    ]
    .rename(
        columns={
            "positive_context": "context",
            "positive_context_id": "context_id"
        }
    )
)

hard_negative_positive_df["similarity"] = np.nan
hard_negative_positive_df["negative_type"] = "positive"

hard_negative_training_df = pd.concat(
    [
        hard_negative_positive_df,
        hard_negative_training_df[
            [
                "id",
                "title",
                "question",
                "context",
                "context_id",
                "answer_text",
                "answer_start",
                "label",
                "similarity",
                "negative_type"
            ]
        ]
    ],
    ignore_index=True
)

hard_negative_training_df = (
    hard_negative_training_df
    .sample(
        frac=1,
        random_state=SEED
    )
    .reset_index(drop=True)
)

print("Balanced hard-negative training dataset")
print(
    "Total pairs:",
    len(hard_negative_training_df)
)
print(
    "Positive pairs:",
    (
        hard_negative_training_df["label"] == 1
    ).sum()
)
print(
    "Hard negative pairs:",
    (
        hard_negative_training_df["label"] == 0
    ).sum()
)
print(
    "Label distribution:",
    hard_negative_training_df["label"]
    .value_counts()
    .to_dict()
)

Balanced hard-negative training dataset
Total pairs: 5610
Positive pairs: 2805
Hard negative pairs: 2805
Label distribution: {0: 2805, 1: 2805}


In [24]:
hard_negative_output_path = (
    "/kaggle/working/week3_day2_hard_negative_training.csv"
)

hard_negative_training_df.to_csv(
    hard_negative_output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Hard-negative training dataset saved to:")
print(hard_negative_output_path)
print("Dataset shape:", hard_negative_training_df.shape)
print(
    hard_negative_training_df["label"]
    .value_counts()
    .sort_index()
)

Hard-negative training dataset saved to:
/kaggle/working/week3_day2_hard_negative_training.csv
Dataset shape: (5610, 10)
label
0    2805
1    2805
Name: count, dtype: int64


# Day 2 Experiment Log

Created a balanced hard-negative dataset for reranker training.


# Day 3 - Cross-Encoder Reranker Baseline Training

Train a Cross-Encoder baseline with random negatives.


In [25]:
# 리랭커 학습 데이터 구성
RERANKER_MAX_LENGTH = 512
RERANKER_BATCH_SIZE = 16

selected_ids = (
    hard_negative_training_df[
        hard_negative_training_df["label"] == 0
    ]["id"]
    .tolist()
)

baseline_positive_df = (
    positive_train_df[
        positive_train_df["id"].isin(selected_ids)
    ]
    .copy()
)

baseline_positive_df = (
    baseline_positive_df[
        [
            "id",
            "title",
            "question",
            "positive_context",
            "positive_context_id",
            "answer_text",
            "answer_start",
            "label"
        ]
    ]
    .rename(
        columns={
            "positive_context": "context",
            "positive_context_id": "context_id"
        }
    )
)

baseline_positive_df["similarity"] = np.nan
baseline_positive_df["negative_type"] = "positive"

baseline_random_negative_df = (
    random_negative_df[
        random_negative_df["id"].isin(selected_ids)
    ]
    .copy()
)

baseline_random_negative_df[
    "similarity"
] = np.nan

random_baseline_df = pd.concat(
    [
        baseline_positive_df,
        baseline_random_negative_df
    ],
    ignore_index=True
)

random_baseline_df = (
    random_baseline_df
    .sample(
        frac=1,
        random_state=SEED
    )
    .reset_index(drop=True)
)

print("Reranker baseline dataset prepared")
print("Total pairs:", len(random_baseline_df))
print(
    "Positive pairs:",
    (random_baseline_df["label"] == 1).sum()
)
print(
    "Random negative pairs:",
    (random_baseline_df["label"] == 0).sum()
)
print(
    "Maximum sequence length:",
    RERANKER_MAX_LENGTH
)
print(
    "Batch size:",
    RERANKER_BATCH_SIZE
)

Reranker baseline dataset prepared
Total pairs: 5610
Positive pairs: 2805
Random negative pairs: 2805
Maximum sequence length: 512
Batch size: 16


In [26]:
validation_positive_records = []

for example in positive_validation_dataset:
    validation_positive_records.append(
        {
            "id": example["id"],
            "title": example["title"],
            "question": example["question"],
            "context": example["context"],
            "context_id": -1,
            "answer_text": example["answer_text"],
            "answer_start": example["answer_start"],
            "label": 1,
            "similarity": np.nan,
            "negative_type": "positive"
        }
    )

validation_positive_df = pd.DataFrame(
    validation_positive_records
)

validation_random_negative_records = []

for _, row in validation_positive_df.iterrows():
    negative_context_id = random.randrange(
        len(unique_contexts)
    )

    while (
        unique_contexts[negative_context_id]
        == row["context"]
    ):
        negative_context_id = random.randrange(
            len(unique_contexts)
        )

    validation_random_negative_records.append(
        {
            "id": row["id"],
            "title": row["title"],
            "question": row["question"],
            "context": unique_contexts[
                negative_context_id
            ],
            "context_id": negative_context_id,
            "answer_text": "",
            "answer_start": -1,
            "label": 0,
            "similarity": np.nan,
            "negative_type": "random"
        }
    )

validation_random_negative_df = pd.DataFrame(
    validation_random_negative_records
)

reranker_validation_df = pd.concat(
    [
        validation_positive_df,
        validation_random_negative_df
    ],
    ignore_index=True
)

reranker_validation_df = (
    reranker_validation_df
    .sample(
        frac=1,
        random_state=SEED
    )
    .reset_index(drop=True)
)

print("Reranker validation dataset prepared")
print("Total pairs:", len(reranker_validation_df))
print(
    "Positive pairs:",
    (reranker_validation_df["label"] == 1).sum()
)
print(
    "Random negative pairs:",
    (reranker_validation_df["label"] == 0).sum()
)

Reranker validation dataset prepared
Total pairs: 11548
Positive pairs: 5774
Random negative pairs: 5774


In [27]:
from transformers import DataCollatorWithPadding

reranker_tokenizer = week2_tokenizer

train_reranker_dataset = Dataset.from_pandas(
    random_baseline_df[
        [
            "question",
            "context",
            "label"
        ]
    ],
    preserve_index=False
)

validation_reranker_dataset = Dataset.from_pandas(
    reranker_validation_df[
        [
            "question",
            "context",
            "label"
        ]
    ],
    preserve_index=False
)


def tokenize_reranker_batch(examples):
    return reranker_tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=RERANKER_MAX_LENGTH,
        padding=False
    )


tokenized_train_reranker = (
    train_reranker_dataset
    .map(
        tokenize_reranker_batch,
        batched=True,
        remove_columns=[
            "question",
            "context"
        ]
    )
    .rename_column(
        "label",
        "labels"
    )
)

tokenized_validation_reranker = (
    validation_reranker_dataset
    .map(
        tokenize_reranker_batch,
        batched=True,
        remove_columns=[
            "question",
            "context"
        ]
    )
    .rename_column(
        "label",
        "labels"
    )
)

data_collator = DataCollatorWithPadding(
    tokenizer=reranker_tokenizer
)

print("Reranker tokenization completed")
print(
    "Train examples:",
    len(tokenized_train_reranker)
)
print(
    "Validation examples:",
    len(tokenized_validation_reranker)
)
print(
    "Train features:",
    tokenized_train_reranker.column_names
)

Map:   0%|          | 0/5610 [00:00<?, ? examples/s]

Map:   0%|          | 0/11548 [00:00<?, ? examples/s]

Reranker tokenization completed
Train examples: 5610
Validation examples: 11548
Train features: ['labels', 'input_ids', 'attention_mask']


In [28]:
from transformers import (
    AutoModelForSequenceClassification
)

RERANKER_MODEL_NAME = "klue/roberta-base"

reranker_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        RERANKER_MODEL_NAME,
        num_labels=2,
        id2label={
            0: "NOT_RELEVANT",
            1: "RELEVANT"
        },
        label2id={
            "NOT_RELEVANT": 0,
            "RELEVANT": 1
        }
    )
    .to(device)
)

total_parameters = sum(
    parameter.numel()
    for parameter in reranker_model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in reranker_model.parameters()
    if parameter.requires_grad
)

print("Cross-Encoder Reranker initialized")
print("Model:", RERANKER_MODEL_NAME)
print("Number of labels:", 2)
print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)
print("Device:", device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Cross-Encoder Reranker initialized
Model: klue/roberta-base
Number of labels: 2
Total parameters: 110619650
Trainable parameters: 110619650
Device: cuda


In [29]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    f1_score
)

def compute_reranker_metrics(eval_prediction):
    logits, labels = eval_prediction

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="binary",
            zero_division=0
        )
    )

    macro_f1 = f1_score(
        labels,
        predictions,
        average="macro"
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "macro_f1": macro_f1
    }

In [30]:
from transformers import (
    TrainingArguments,
    Trainer
)

RERANKER_LEARNING_RATE = 2e-5
RERANKER_EPOCHS = 1
RERANKER_WEIGHT_DECAY = 0.01

reranker_training_args = TrainingArguments(
    output_dir="./week3_day3_reranker_baseline",
    learning_rate=RERANKER_LEARNING_RATE,
    per_device_train_batch_size=RERANKER_BATCH_SIZE,
    per_device_eval_batch_size=RERANKER_BATCH_SIZE,
    num_train_epochs=RERANKER_EPOCHS,
    weight_decay=RERANKER_WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
    data_seed=SEED
)

reranker_trainer = Trainer(
    model=reranker_model,
    args=reranker_training_args,
    train_dataset=tokenized_train_reranker,
    eval_dataset=tokenized_validation_reranker,
    processing_class=reranker_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_reranker_metrics
)

print("Reranker baseline trainer is ready")
print("Learning rate:", RERANKER_LEARNING_RATE)
print("Epochs:", RERANKER_EPOCHS)
print("Batch size:", RERANKER_BATCH_SIZE)
print("Weight decay:", RERANKER_WEIGHT_DECAY)
print("Max length:", RERANKER_MAX_LENGTH)

Reranker baseline trainer is ready
Learning rate: 2e-05
Epochs: 1
Batch size: 16
Weight decay: 0.01
Max length: 512


In [31]:
print("Starting reranker baseline training")

reranker_train_result = reranker_trainer.train()

print("Reranker baseline training completed")
print(
    "Training loss:",
    f"{reranker_train_result.training_loss:.4f}"
)
print(
    "Training runtime:",
    f"{reranker_train_result.metrics['train_runtime']:.2f} seconds"
)
print(
    "Training samples per second:",
    f"{reranker_train_result.metrics['train_samples_per_second']:.2f}"
)

Starting reranker baseline training


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Macro F1
1,0.066986,0.079291,0.992466,0.999122,0.985798,0.992416,0.992466


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Reranker baseline training completed
Training loss: 0.2799
Training runtime: 532.21 seconds
Training samples per second: 10.54


In [32]:
baseline_metrics = reranker_trainer.evaluate()

print("Reranker baseline evaluation completed")

for metric_name, metric_value in baseline_metrics.items():
    print(f"{metric_name}: {metric_value}")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Reranker baseline evaluation completed
eval_loss: 0.07929062098264694
eval_accuracy: 0.9924662279182542
eval_precision: 0.9991223450939091
eval_recall: 0.9857984066505022
eval_f1: 0.9924156568738558
eval_macro_f1: 0.992465892953057
eval_runtime: 212.9143
eval_samples_per_second: 54.238
eval_steps_per_second: 1.696
epoch: 1.0


In [33]:
baseline_artifact_dir = "/kaggle/working/week3_day3_reranker_baseline"

reranker_trainer.save_model(baseline_artifact_dir)
reranker_tokenizer.save_pretrained(baseline_artifact_dir)

print("Baseline model saved to:", baseline_artifact_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Baseline model saved to: /kaggle/working/week3_day3_reranker_baseline


# Day 3 Experiment Log

Verified the baseline Cross-Encoder training loop with random negatives.


# Day 4 - Hard-Negative Reranker Fine-tuning and Hyperparameter Comparison

Fine-tune with hard negatives and select the final reranker.


In [34]:
import pandas as pd

from sklearn.model_selection import train_test_split
from datasets import Dataset

hard_negative_input_path = (
    "/kaggle/working/"
    "week3_day2_hard_negative_training.csv"
)

hard_negative_training_df = pd.read_csv(
    hard_negative_input_path,
    encoding="utf-8-sig"
)

hard_negative_pair_df = (
    hard_negative_training_df[
        ["id", "question", "context", "label"]
    ]
    .copy()
)

hard_negative_pair_df["label"] = (
    hard_negative_pair_df["label"].astype(int)
)

unique_question_ids = (
    hard_negative_pair_df["id"]
    .drop_duplicates()
    .tolist()
)

train_ids, validation_ids = train_test_split(
    unique_question_ids,
    test_size=0.2,
    random_state=378
)

hard_negative_train_df = (
    hard_negative_pair_df[
        hard_negative_pair_df["id"].isin(train_ids)
    ]
    .reset_index(drop=True)
)

hard_negative_validation_df = (
    hard_negative_pair_df[
        hard_negative_pair_df["id"].isin(validation_ids)
    ]
    .reset_index(drop=True)
)

hard_negative_train_dataset = Dataset.from_pandas(
    hard_negative_train_df,
    preserve_index=False
)

hard_negative_validation_dataset = Dataset.from_pandas(
    hard_negative_validation_df,
    preserve_index=False
)

def tokenize_hard_negative_pairs(batch):
    return reranker_tokenizer(
        batch["question"],
        batch["context"],
        truncation="only_second",
        max_length=512
    )

hard_negative_train_tokenized = (
    hard_negative_train_dataset.map(
        tokenize_hard_negative_pairs,
        batched=True,
        remove_columns=[
            "id",
            "question",
            "context"
        ]
    )
)

hard_negative_validation_tokenized = (
    hard_negative_validation_dataset.map(
        tokenize_hard_negative_pairs,
        batched=True,
        remove_columns=[
            "id",
            "question",
            "context"
        ]
    )
)

print("Hard-negative datasets prepared")
print(
    "Training pairs:",
    len(hard_negative_train_tokenized)
)
print(
    "Validation pairs:",
    len(hard_negative_validation_tokenized)
)
print(
    "Validation features:",
    hard_negative_validation_tokenized.column_names
)

Map:   0%|          | 0/4487 [00:00<?, ? examples/s]

Map:   0%|          | 0/1123 [00:00<?, ? examples/s]

Hard-negative datasets prepared
Training pairs: 4487
Validation pairs: 1123
Validation features: ['label', 'input_ids', 'attention_mask']


In [35]:
hard_negative_baseline_metrics = (
    reranker_trainer.evaluate(
        eval_dataset=hard_negative_validation_tokenized,
        metric_key_prefix="hard_negative_baseline"
    )
)

print("Hard-negative baseline evaluation completed")

for metric_name, metric_value in (
    hard_negative_baseline_metrics.items()
):
    print(f"{metric_name}: {metric_value}")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Hard-negative baseline evaluation completed
hard_negative_baseline_loss: 2.289438247680664
hard_negative_baseline_accuracy: 0.8032056990204809
hard_negative_baseline_precision: 0.7348484848484849
hard_negative_baseline_recall: 0.9814502529510961
hard_negative_baseline_f1: 0.8404332129963898
hard_negative_baseline_macro_f1: 0.7918774659639325
hard_negative_baseline_runtime: 20.4716
hard_negative_baseline_samples_per_second: 54.857
hard_negative_baseline_steps_per_second: 1.759
epoch: 1.0


In [36]:
import json

hard_negative_baseline_result_path = (
    "/kaggle/working/"
    "week3_day4_hard_negative_baseline_results.json"
)

with open(
    hard_negative_baseline_result_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        hard_negative_baseline_metrics,
        file,
        ensure_ascii=False,
        indent=2
    )

print(
    "Hard-negative baseline results saved to:",
    hard_negative_baseline_result_path
)

Hard-negative baseline results saved to: /kaggle/working/week3_day4_hard_negative_baseline_results.json


In [37]:
import gc
import os
import time
import torch
import pandas as pd

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

LEARNING_RATES = [1e-5, 3e-5, 5e-5]

WEIGHT_DECAY = 0.01
BATCH_SIZE = 16
MAX_EPOCHS = 5
RANDOM_SEED = 378

experiment_results = []
experiment_logs = []

data_collator = DataCollatorWithPadding(
    tokenizer=reranker_tokenizer
)

for learning_rate in LEARNING_RATES:
    experiment_name = f"lr_{learning_rate:.0e}"
    experiment_output_dir = (
        f"/kaggle/working/week3_day4_{experiment_name}"
    )

    print("Starting experiment:", experiment_name)
    print("Learning rate:", learning_rate)
    print("Weight decay:", WEIGHT_DECAY)
    print("Maximum epochs:", MAX_EPOCHS)

    start_time = time.time()

    experiment_model = (
        AutoModelForSequenceClassification.from_pretrained(
            "klue/roberta-base",
            num_labels=2
        )
    )

    training_args = TrainingArguments(
        output_dir=experiment_output_dir,
        learning_rate=learning_rate,
        weight_decay=WEIGHT_DECAY,
        num_train_epochs=MAX_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_macro_f1",
        greater_is_better=True,
        save_total_limit=1,
        fp16=torch.cuda.is_available(),
        seed=RANDOM_SEED,
        report_to="none"
    )

    experiment_trainer = Trainer(
        model=experiment_model,
        args=training_args,
        train_dataset=hard_negative_train_tokenized,
        eval_dataset=hard_negative_validation_tokenized,
        data_collator=data_collator,
        processing_class=reranker_tokenizer,
        compute_metrics=compute_reranker_metrics
    )

    train_result = experiment_trainer.train()

    evaluation_metrics = experiment_trainer.evaluate()

    elapsed_time = time.time() - start_time

    epoch_records = [
        log_record
        for log_record in experiment_trainer.state.log_history
        if "eval_macro_f1" in log_record
    ]

    for log_record in epoch_records:
        experiment_logs.append(
            {
                "experiment": experiment_name,
                "learning_rate": learning_rate,
                "epoch": log_record.get("epoch"),
                "eval_loss": log_record.get("eval_loss"),
                "eval_accuracy": log_record.get(
                    "eval_accuracy"
                ),
                "eval_precision": log_record.get(
                    "eval_precision"
                ),
                "eval_recall": log_record.get(
                    "eval_recall"
                ),
                "eval_f1": log_record.get("eval_f1"),
                "eval_macro_f1": log_record.get(
                    "eval_macro_f1"
                )
            }
        )

    if epoch_records:
        best_epoch_record = max(
            epoch_records,
            key=lambda record: record["eval_macro_f1"]
        )

        best_epoch = best_epoch_record.get("epoch")
        best_macro_f1 = best_epoch_record.get(
            "eval_macro_f1"
        )
        best_eval_loss = best_epoch_record.get(
            "eval_loss"
        )
    else:
        best_epoch = None
        best_macro_f1 = evaluation_metrics.get(
            "eval_macro_f1"
        )
        best_eval_loss = evaluation_metrics.get(
            "eval_loss"
        )

    experiment_result = {
        "experiment": experiment_name,
        "learning_rate": learning_rate,
        "best_epoch": best_epoch,
        "best_macro_f1": best_macro_f1,
        "best_eval_loss": best_eval_loss,
        "final_eval_macro_f1": evaluation_metrics.get(
            "eval_macro_f1"
        ),
        "final_eval_accuracy": evaluation_metrics.get(
            "eval_accuracy"
        ),
        "final_eval_precision": evaluation_metrics.get(
            "eval_precision"
        ),
        "final_eval_recall": evaluation_metrics.get(
            "eval_recall"
        ),
        "runtime_minutes": elapsed_time / 60,
        "best_checkpoint": experiment_trainer.state.best_model_checkpoint
    }

    experiment_results.append(experiment_result)

    print("Experiment completed:", experiment_name)
    print("Best epoch:", best_epoch)
    print(
        "Best validation Macro F1:",
        f"{best_macro_f1:.6f}"
    )
    print(
        "Best validation loss:",
        f"{best_eval_loss:.6f}"
    )
    print(
        "Runtime:",
        f"{elapsed_time / 60:.2f} minutes"
    )
    print(
        "Best checkpoint:",
        experiment_trainer.state.best_model_checkpoint
    )

    del experiment_trainer
    del experiment_model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_df = pd.DataFrame(experiment_results)
epoch_logs_df = pd.DataFrame(experiment_logs)

results_df = results_df.sort_values(
    by="best_macro_f1",
    ascending=False
).reset_index(drop=True)

print("Learning-rate experiment summary")

display(results_df)

print("Epoch-level validation results")

display(epoch_logs_df)

results_df.to_csv(
    "/kaggle/working/week3_day4_learning_rate_results.csv",
    index=False,
    encoding="utf-8-sig"
)

epoch_logs_df.to_csv(
    "/kaggle/working/week3_day4_epoch_logs.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Summary saved to:",
    "/kaggle/working/week3_day4_learning_rate_results.csv"
)

print(
    "Epoch logs saved to:",
    "/kaggle/working/week3_day4_epoch_logs.csv"
)

Starting experiment: lr_1e-05
Learning rate: 1e-05
Weight decay: 0.01
Maximum epochs: 5


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather al

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Macro F1
1,1.102286,0.342329,0.947462,0.926518,0.978078,0.951600,0.947075
2,0.270441,0.260202,0.958148,0.962712,0.957841,0.960270,0.958028
3,0.128091,0.236010,0.973286,0.968386,0.981450,0.974874,0.973179
4,0.070316,0.303073,0.967053,0.976027,0.961214,0.968564,0.966976
5,0.046875,0.301327,0.970614,0.962046,0.983137,0.972477,0.970479


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Experiment completed: lr_5e-05
Best epoch: 5.0
Best validation Macro F1: 0.969573
Best validation loss: 0.397463
Runtime: 22.91 minutes
Best checkpoint: /kaggle/working/week3_day4_lr_5e-05/checkpoint-705
Learning-rate experiment summary


,experiment,learning_rate,best_epoch,best_macro_f1,best_eval_loss,final_eval_macro_f1,final_eval_accuracy,final_eval_precision,final_eval_recall,runtime_minutes,best_checkpoint
0,lr_3e-05,0.00003,5.0,0.973196,0.356248,0.973196,0.973286,0.973109,0.976391,22.875860,/kaggle/working/week3_day4_lr_3e-05/checkpoint...
1,lr_1e-05,0.00001,3.0,0.973179,0.236010,0.973179,0.973286,0.968386,0.981450,22.945119,/kaggle/working/week3_day4_lr_1e-05/checkpoint...
2,lr_5e-05,0.00005,5.0,0.969573,0.397463,0.969573,0.969724,0.958949,0.984823,22.912285,/kaggle/working/week3_day4_lr_5e-05/checkpoint...


Epoch-level validation results


,experiment,learning_rate,epoch,eval_loss,eval_accuracy,eval_precision,eval_recall,eval_f1,eval_macro_f1
0,lr_1e-05,0.00001,1.0,0.342329,0.947462,0.926518,0.978078,0.951600,0.947075
1,lr_1e-05,0.00001,2.0,0.260202,0.958148,0.962712,0.957841,0.960270,0.958028
2,lr_1e-05,0.00001,3.0,0.236010,0.973286,0.968386,0.981450,0.974874,0.973179
3,lr_1e-05,0.00001,4.0,0.303073,0.967053,0.976027,0.961214,0.968564,0.966976
4,lr_1e-05,0.00001,5.0,0.301327,0.970614,0.962046,0.983137,0.972477,0.970479
5,lr_1e-05,0.00001,5.0,0.236117,0.973286,0.968386,0.981450,0.974874,0.973179
6,lr_3e-05,0.00003,1.0,0.296920,0.954586,0.928797,0.989882,0.958367,0.954208
7,lr_3e-05,0.00003,2.0,0.295495,0.969724,0.955954,0.988196,0.971808,0.969558
8,lr_3e-05,0.00003,3.0,0.317773,0.964381,0.982548,0.949410,0.965695,0.964329
9,lr_3e-05,0.00003,4.0,0.343242,0.970614,0.974576,0.969646,0.972105,0.970530


Summary saved to: /kaggle/working/week3_day4_learning_rate_results.csv
Epoch logs saved to: /kaggle/working/week3_day4_epoch_logs.csv


In [38]:
import gc
import os
import time
import torch
import pandas as pd

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

FIXED_LEARNING_RATE = 1e-5
WEIGHT_DECAYS = [0.05, 0.1]

BATCH_SIZE = 16
MAX_EPOCHS = 5
RANDOM_SEED = 378

weight_decay_results = []
weight_decay_logs = []

data_collator = DataCollatorWithPadding(
    tokenizer=reranker_tokenizer
)

for weight_decay in WEIGHT_DECAYS:
    experiment_name = (
        f"lr_1e-5_wd_{weight_decay}"
    )

    experiment_output_dir = (
        f"/kaggle/working/week3_day4_{experiment_name}"
    )

    print("Starting experiment:", experiment_name)
    print("Learning rate:", FIXED_LEARNING_RATE)
    print("Weight decay:", weight_decay)
    print("Maximum epochs:", MAX_EPOCHS)

    start_time = time.time()

    experiment_model = (
        AutoModelForSequenceClassification.from_pretrained(
            "klue/roberta-base",
            num_labels=2
        )
    )

    training_args = TrainingArguments(
        output_dir=experiment_output_dir,
        learning_rate=FIXED_LEARNING_RATE,
        weight_decay=weight_decay,
        num_train_epochs=MAX_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_macro_f1",
        greater_is_better=True,
        save_total_limit=1,
        fp16=torch.cuda.is_available(),
        seed=RANDOM_SEED,
        report_to="none"
    )

    experiment_trainer = Trainer(
        model=experiment_model,
        args=training_args,
        train_dataset=hard_negative_train_tokenized,
        eval_dataset=hard_negative_validation_tokenized,
        data_collator=data_collator,
        processing_class=reranker_tokenizer,
        compute_metrics=compute_reranker_metrics
    )

    train_result = experiment_trainer.train()

    evaluation_metrics = (
        experiment_trainer.evaluate()
    )

    elapsed_time = time.time() - start_time

    epoch_records = [
        log_record
        for log_record in experiment_trainer.state.log_history
        if "eval_macro_f1" in log_record
    ]

    for log_record in epoch_records:
        weight_decay_logs.append(
            {
                "experiment": experiment_name,
                "learning_rate": FIXED_LEARNING_RATE,
                "weight_decay": weight_decay,
                "epoch": log_record.get("epoch"),
                "eval_loss": log_record.get("eval_loss"),
                "eval_accuracy": log_record.get(
                    "eval_accuracy"
                ),
                "eval_precision": log_record.get(
                    "eval_precision"
                ),
                "eval_recall": log_record.get(
                    "eval_recall"
                ),
                "eval_f1": log_record.get(
                    "eval_f1"
                ),
                "eval_macro_f1": log_record.get(
                    "eval_macro_f1"
                )
            }
        )

    if epoch_records:
        best_epoch_record = max(
            epoch_records,
            key=lambda record: record["eval_macro_f1"]
        )

        best_epoch = best_epoch_record.get("epoch")
        best_macro_f1 = best_epoch_record.get(
            "eval_macro_f1"
        )
        best_eval_loss = best_epoch_record.get(
            "eval_loss"
        )
    else:
        best_epoch = None
        best_macro_f1 = evaluation_metrics.get(
            "eval_macro_f1"
        )
        best_eval_loss = evaluation_metrics.get(
            "eval_loss"
        )

    weight_decay_results.append(
        {
            "experiment": experiment_name,
            "learning_rate": FIXED_LEARNING_RATE,
            "weight_decay": weight_decay,
            "best_epoch": best_epoch,
            "best_macro_f1": best_macro_f1,
            "best_eval_loss": best_eval_loss,
            "final_eval_macro_f1": evaluation_metrics.get(
                "eval_macro_f1"
            ),
            "final_eval_accuracy": evaluation_metrics.get(
                "eval_accuracy"
            ),
            "final_eval_precision": evaluation_metrics.get(
                "eval_precision"
            ),
            "final_eval_recall": evaluation_metrics.get(
                "eval_recall"
            ),
            "runtime_minutes": elapsed_time / 60,
            "best_checkpoint": (
                experiment_trainer
                .state
                .best_model_checkpoint
            )
        }
    )

    print("Experiment completed:", experiment_name)
    print("Best epoch:", best_epoch)
    print(
        "Best validation Macro F1:",
        f"{best_macro_f1:.6f}"
    )
    print(
        "Best validation loss:",
        f"{best_eval_loss:.6f}"
    )
    print(
        "Runtime:",
        f"{elapsed_time / 60:.2f} minutes"
    )
    print(
        "Best checkpoint:",
        experiment_trainer.state.best_model_checkpoint
    )

    del experiment_trainer
    del experiment_model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

weight_decay_results_df = pd.DataFrame(
    weight_decay_results
)

weight_decay_logs_df = pd.DataFrame(
    weight_decay_logs
)

weight_decay_results_df = (
    weight_decay_results_df
    .sort_values(
        by="best_macro_f1",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Weight decay experiment summary")

display(weight_decay_results_df)

print("Epoch-level validation results")

display(weight_decay_logs_df)

weight_decay_results_df.to_csv(
    "/kaggle/working/"
    "week3_day4_weight_decay_results.csv",
    index=False,
    encoding="utf-8-sig"
)

weight_decay_logs_df.to_csv(
    "/kaggle/working/"
    "week3_day4_weight_decay_epoch_logs.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Summary saved to:",
    "/kaggle/working/"
    "week3_day4_weight_decay_results.csv"
)

print(
    "Epoch logs saved to:",
    "/kaggle/working/"
    "week3_day4_weight_decay_epoch_logs.csv"
)

Starting experiment: lr_1e-5_wd_0.05
Learning rate: 1e-05
Weight decay: 0.05
Maximum epochs: 5


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather al

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Macro F1
1,0.956116,0.393462,0.946572,0.913178,0.993255,0.951535,0.946005
2,0.228307,0.276395,0.964381,0.954023,0.979764,0.966722,0.964204
3,0.098735,0.265799,0.967943,0.963394,0.976391,0.969849,0.967814
4,0.046872,0.260786,0.975957,0.971667,0.983137,0.977368,0.975863
5,0.030914,0.272265,0.975957,0.968543,0.986509,0.977444,0.975852


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Experiment completed: lr_1e-5_wd_0.1
Best epoch: 3.0
Best validation Macro F1: 0.975007
Best validation loss: 0.248895
Runtime: 22.95 minutes
Best checkpoint: /kaggle/working/week3_day4_lr_1e-5_wd_0.1/checkpoint-423
Weight decay experiment summary


,experiment,learning_rate,weight_decay,best_epoch,best_macro_f1,best_eval_loss,final_eval_macro_f1,final_eval_accuracy,final_eval_precision,final_eval_recall,runtime_minutes,best_checkpoint
0,lr_1e-5_wd_0.05,0.00001,0.05,4.0,0.975863,0.260786,0.975863,0.975957,0.971667,0.983137,22.906498,/kaggle/working/week3_day4_lr_1e-5_wd_0.05/che...
1,lr_1e-5_wd_0.1,0.00001,0.10,3.0,0.975007,0.248895,0.975007,0.975067,0.982906,0.969646,22.953986,/kaggle/working/week3_day4_lr_1e-5_wd_0.1/chec...


Epoch-level validation results


,experiment,learning_rate,weight_decay,epoch,eval_loss,eval_accuracy,eval_precision,eval_recall,eval_f1,eval_macro_f1
0,lr_1e-5_wd_0.05,0.00001,0.05,1.0,0.393462,0.946572,0.913178,0.993255,0.951535,0.946005
1,lr_1e-5_wd_0.05,0.00001,0.05,2.0,0.276395,0.964381,0.954023,0.979764,0.966722,0.964204
2,lr_1e-5_wd_0.05,0.00001,0.05,3.0,0.265799,0.967943,0.963394,0.976391,0.969849,0.967814
3,lr_1e-5_wd_0.05,0.00001,0.05,4.0,0.260786,0.975957,0.971667,0.983137,0.977368,0.975863
4,lr_1e-5_wd_0.05,0.00001,0.05,5.0,0.272265,0.975957,0.968543,0.986509,0.977444,0.975852
5,lr_1e-5_wd_0.05,0.00001,0.05,5.0,0.260812,0.975957,0.971667,0.983137,0.977368,0.975863
6,lr_1e-5_wd_0.1,0.00001,0.10,1.0,0.373806,0.942119,0.907407,0.991568,0.947623,0.941473
7,lr_1e-5_wd_0.1,0.00001,0.10,2.0,0.265672,0.968833,0.954397,0.988196,0.971002,0.968658
8,lr_1e-5_wd_0.1,0.00001,0.10,3.0,0.248895,0.975067,0.982906,0.969646,0.976231,0.975007
9,lr_1e-5_wd_0.1,0.00001,0.10,4.0,0.261947,0.972395,0.973064,0.974705,0.973884,0.972305


Summary saved to: /kaggle/working/week3_day4_weight_decay_results.csv
Epoch logs saved to: /kaggle/working/week3_day4_weight_decay_epoch_logs.csv


# Day 4 Experiment Log

The learning-rate screen reached Macro-F1 0.9732 at 3e-5. The persisted final checkpoint was retrained with learning rate 1e-5 and reached validation Macro-F1 0.9750.


# Day 5 - Dense Retriever-based Reranking Candidate Generation

Generate dense-retrieval candidates for reranking.


In [39]:
import os
import random
import numpy as np
import pandas as pd
import faiss

from pathlib import Path

DAY5_RANDOM_SEED = 378
DAY5_QUERY_LIMIT = 1000
DAY5_RETRIEVAL_K = 20
DAY5_FINAL_CANDIDATE_K = 10

random.seed(DAY5_RANDOM_SEED)
np.random.seed(DAY5_RANDOM_SEED)

if "korquad" not in globals():
    raise NameError(
        "korquad is not available. "
        "Load the KorQuAD dataset first."
    )

if "week2_encoder" not in globals():
    raise NameError(
        "week2_encoder is not available. "
        "Load the Week 2 dense retriever first."
    )

validation_records = []

for example in korquad["validation"]:
    answer_text = example["answers"]["text"][0]
    answer_start = example["answers"]["answer_start"][0]

    validation_records.append(
        {
            "id": example["id"],
            "title": example["title"],
            "question": example["question"],
            "context": example["context"],
            "answer_text": answer_text,
            "answer_start": answer_start
        }
    )

validation_df = pd.DataFrame(validation_records)

validation_query_df = (
    validation_df
    .sample(
        n=min(DAY5_QUERY_LIMIT, len(validation_df)),
        random_state=DAY5_RANDOM_SEED
    )
    .reset_index(drop=True)
)

corpus_df = (
    validation_df[
        ["title", "context"]
    ]
    .drop_duplicates(subset=["context"])
    .reset_index(drop=True)
)

context_to_corpus_index = {
    context: index
    for index, context in enumerate(corpus_df["context"])
}

validation_query_df["gold_corpus_index"] = (
    validation_query_df["context"]
    .map(context_to_corpus_index)
)

print("Week 3 - Day 5 Retrieval Setup")
print("Number of queries:", len(validation_query_df))
print("Corpus size:", len(corpus_df))
print("Retrieval K:", DAY5_RETRIEVAL_K)
print("Final candidate K:", DAY5_FINAL_CANDIDATE_K)

Week 3 - Day 5 Retrieval Setup
Number of queries: 1000
Corpus size: 960
Retrieval K: 20
Final candidate K: 10


In [40]:
query_texts = validation_query_df["question"].tolist()
corpus_texts = corpus_df["context"].tolist()

print("Encoding query texts...")

query_embeddings = week2_encoder.encode(
    query_texts
)

print("Query embeddings completed")
print("Query embedding shape:", query_embeddings.shape)

print("Encoding corpus contexts...")

corpus_embeddings = week2_encoder.encode(
    corpus_texts
)

print("Corpus embeddings completed")
print("Corpus embedding shape:", corpus_embeddings.shape)

query_embeddings = np.asarray(
    query_embeddings,
    dtype="float32"
)

corpus_embeddings = np.asarray(
    corpus_embeddings,
    dtype="float32"
)

faiss.normalize_L2(query_embeddings)
faiss.normalize_L2(corpus_embeddings)

retrieval_index = faiss.IndexFlatIP(
    corpus_embeddings.shape[1]
)

retrieval_index.add(corpus_embeddings)

dense_similarities, dense_indices = (
    retrieval_index.search(
        query_embeddings,
        DAY5_RETRIEVAL_K
    )
)

print("Dense retrieval completed")
print("Similarity matrix shape:", dense_similarities.shape)
print("Retrieved index shape:", dense_indices.shape)
print("First query retrieved indices:", dense_indices[0])
print("First query similarities:", dense_similarities[0])

Encoding query texts...
Query embeddings completed
Query embedding shape: (1000, 768)
Encoding corpus contexts...
Corpus embeddings completed
Corpus embedding shape: (960, 768)
Dense retrieval completed
Similarity matrix shape: (1000, 20)
Retrieved index shape: (1000, 20)
First query retrieved indices: [270 257 253 456 271 831 228 476 855  27 486 651 854  28 165   2 540 528
 285 263]
First query similarities: [0.5918989  0.55704665 0.47036508 0.46901026 0.4529131  0.44124
 0.43918315 0.43702894 0.43689165 0.42692596 0.42311627 0.42143935
 0.41702214 0.41637903 0.41425714 0.40699768 0.4062416  0.4022888
 0.40209725 0.40209725]


In [41]:
candidate_records = []

for query_index, query_row in validation_query_df.iterrows():
    query_id = query_row["id"]
    question = query_row["question"]
    gold_context = query_row["context"]
    gold_corpus_index = query_row["gold_corpus_index"]
    answer_text = str(query_row["answer_text"])

    candidate_records.append(
        {
            "query_id": query_id,
            "question": question,
            "title": query_row["title"],
            "context": gold_context,
            "label": 1,
            "candidate_rank": 0,
            "dense_rank": 0,
            "dense_similarity": 1.0,
            "negative_type": "positive"
        }
    )

    negative_count = 0

    for dense_rank, (
        corpus_index,
        similarity
    ) in enumerate(
        zip(
            dense_indices[query_index],
            dense_similarities[query_index]
        ),
        start=1
    ):
        corpus_index = int(corpus_index)
        similarity = float(similarity)

        if corpus_index == int(gold_corpus_index):
            continue

        retrieved_context = corpus_df.iloc[
            corpus_index
        ]["context"]

        if answer_text and answer_text in retrieved_context:
            continue

        candidate_records.append(
            {
                "query_id": query_id,
                "question": question,
                "title": corpus_df.iloc[
                    corpus_index
                ]["title"],
                "context": retrieved_context,
                "label": 0,
                "candidate_rank": negative_count + 1,
                "dense_rank": dense_rank,
                "dense_similarity": similarity,
                "negative_type": "dense_hard_negative"
            }
        )

        negative_count += 1

        if negative_count >= DAY5_FINAL_CANDIDATE_K - 1:
            break

candidate_df = pd.DataFrame(candidate_records)

print("Day 5 candidate dataset created")
print("Candidate dataset shape:", candidate_df.shape)
print(
    "Number of unique queries:",
    candidate_df["query_id"].nunique()
)
print(
    "Label distribution:",
    candidate_df["label"].value_counts().to_dict()
)
print(
    "Negative type distribution:",
    candidate_df["negative_type"]
    .value_counts()
    .to_dict()
)

Day 5 candidate dataset created
Candidate dataset shape: (9986, 9)
Number of unique queries: 1000
Label distribution: {0: 8986, 1: 1000}
Negative type distribution: {'dense_hard_negative': 8986, 'positive': 1000}


In [42]:
day5_candidate_path = (
    "/kaggle/working/"
    "week3_day5_dense_reranking_candidates.csv"
)

candidate_df.to_csv(
    day5_candidate_path,
    index=False,
    encoding="utf-8-sig"
)

print("Day 5 candidate dataset saved to:")
print(day5_candidate_path)

print(candidate_df.head())

Day 5 candidate dataset saved to:
/kaggle/working/week3_day5_dense_reranking_candidates.csv
       query_id            question title  \
0  6589969-27-0  이명박은 언제 취임식을 가졌는가?   이명박   
1  6589969-27-0  이명박은 언제 취임식을 가졌는가?   이명박   
2  6589969-27-0  이명박은 언제 취임식을 가졌는가?   이회창   
3  6589969-27-0  이명박은 언제 취임식을 가졌는가?   이명박   
4  6589969-27-0  이명박은 언제 취임식을 가졌는가?   김영삼   

                                             context  label  candidate_rank  \
0  2008년 2월 25일 이명박은 취임식과 함께 업무 수행을 시작했다. 새 정부의 이...      1               0   
1  폭로전은 경선 때부터 시작되었다. 전당 대회를 3일 앞둔 2007년 8월 17일, ...      0               1   
2  2009년 12월 7일 불교방송 ‘김재원의 아침저널’에 나와 “4대강은 이 대통령 ...      0               2   
3  그러나 이명박은 첫판부터 위기를 맞이하게 된다. 당시 논란이 된 광우병 쇠고기 문제...      0               3   
4  대학 2학년에 재학 중 김영삼은 서울특별시 명동 시공관에서 있은 정부수립기념 웅변대...      0               4   

   dense_rank  dense_similarity        negative_type  
0           0          1.000000             positive  
1           3          0.470365  dense_hard_negative

In [43]:
gold_in_top20 = []

for query_index, query_row in validation_query_df.iterrows():
    gold_corpus_index = int(query_row["gold_corpus_index"])
    retrieved_indices = set(dense_indices[query_index].tolist())
    gold_in_top20.append(gold_corpus_index in retrieved_indices)

gold_retrieval_coverage_top20 = np.mean(gold_in_top20)

print("Dense Retriever Top-20 Coverage")
print("Queries:", len(gold_in_top20))
print("Gold contexts found in Top-20:", sum(gold_in_top20))
print("Gold contexts not found in Top-20:", len(gold_in_top20) - sum(gold_in_top20))
print("Top-20 coverage:", f"{gold_retrieval_coverage_top20:.4f}")


Dense Retriever Top-20 Coverage
Queries: 1000
Gold contexts found in Top-20: 818
Gold contexts not found in Top-20: 182
Top-20 coverage: 0.8180


In [44]:
ranking_candidate_records = []

for query_index, query_row in validation_query_df.iterrows():
    query_id = query_row["id"]
    question = query_row["question"]
    gold_corpus_index = int(query_row["gold_corpus_index"])
    retrieved_indices = dense_indices[query_index]
    retrieved_similarities = dense_similarities[query_index]
    gold_found_top20 = gold_corpus_index in retrieved_indices.tolist()

    for rank, (corpus_index, similarity) in enumerate(
        zip(retrieved_indices, retrieved_similarities), start=1
    ):
        corpus_index = int(corpus_index)
        ranking_candidate_records.append(
            {
                "query_id": query_id,
                "question": question,
                "title": corpus_df.iloc[corpus_index]["title"],
                "context": corpus_df.iloc[corpus_index]["context"],
                "label": int(corpus_index == gold_corpus_index),
                "candidate_rank": rank,
                "dense_rank": rank,
                "dense_similarity": float(similarity),
                "gold_in_top20": gold_found_top20,
                "negative_type": "positive" if corpus_index == gold_corpus_index else "dense_hard_negative",
            }
        )

ranking_candidate_df = pd.DataFrame(ranking_candidate_records)

print("Top-20 ranking candidate dataset created")
print("Dataset shape:", ranking_candidate_df.shape)
print("Number of queries:", ranking_candidate_df["query_id"].nunique())
print("Positive candidates:", ranking_candidate_df["label"].sum())
print("Queries with gold in Top-20:", ranking_candidate_df["gold_in_top20"].drop_duplicates().sum())
print("Candidate type distribution:", ranking_candidate_df["negative_type"].value_counts().to_dict())


Top-20 ranking candidate dataset created
Dataset shape: (20000, 10)
Number of queries: 1000
Positive candidates: 818
Queries with gold in Top-20: 1
Candidate type distribution: {'dense_hard_negative': 19182, 'positive': 818}


In [45]:
ranking_candidate_path = "/kaggle/working/week3_day5_ranking_top20_candidates.csv"
ranking_candidate_df.to_csv(ranking_candidate_path, index=False, encoding="utf-8-sig")
print("Ranking candidate dataset saved to:", ranking_candidate_path)


Ranking candidate dataset saved to: /kaggle/working/week3_day5_ranking_top20_candidates.csv


# Day 5 Experiment Log

Generated the fixed Top-20 candidate set used for ranking evaluation.


# Day 6 - Recall/MRR/nDCG Evaluation Before and After Reranking

Measure ranking quality before and after reranking.


In [46]:
import json
import time
from pathlib import Path
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments

FINAL_MODEL_DIR = Path("/kaggle/working/week3_day4_final_reranker")
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
final_model = AutoModelForSequenceClassification.from_pretrained("klue/roberta-base", num_labels=2)
final_args = TrainingArguments(
    output_dir=str(FINAL_MODEL_DIR), learning_rate=1e-5, weight_decay=0.01,
    num_train_epochs=5, per_device_train_batch_size=16, per_device_eval_batch_size=16,
    eval_strategy="epoch", save_strategy="epoch", logging_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="eval_macro_f1",
    greater_is_better=True, save_total_limit=2, fp16=torch.cuda.is_available(),
    seed=SEED, report_to="none",
)
final_trainer = Trainer(
    model=final_model, args=final_args,
    train_dataset=hard_negative_train_tokenized,
    eval_dataset=hard_negative_validation_tokenized,
    data_collator=DataCollatorWithPadding(tokenizer=reranker_tokenizer),
    processing_class=reranker_tokenizer,
    compute_metrics=compute_reranker_metrics,
)
started = time.time()
final_trainer.train()
final_metrics = final_trainer.evaluate()
final_trainer.save_model(str(FINAL_MODEL_DIR))
reranker_tokenizer.save_pretrained(str(FINAL_MODEL_DIR))
with (FINAL_MODEL_DIR / "final_reranker_metrics.json").open("w", encoding="utf-8") as file:
    json.dump(final_metrics, file, ensure_ascii=False, indent=2, default=float)
print("Final reranker saved:", FINAL_MODEL_DIR)
print("Training minutes:", f"{(time.time() - started) / 60:.2f}")
print("Validation Macro-F1:", f"{final_metrics['eval_macro_f1']:.4f}")


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather al

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Macro F1
1,0.955770,0.312918,0.955476,0.934400,0.984823,0.958949,0.955155
2,0.228568,0.269802,0.966162,0.954173,0.983137,0.968439,0.965985
3,0.118651,0.237694,0.972395,0.971477,0.976391,0.973928,0.972300
4,0.052389,0.262221,0.972395,0.971477,0.976391,0.973928,0.972300
5,0.034958,0.269415,0.975067,0.966942,0.986509,0.976628,0.974955


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final reranker saved: /kaggle/working/week3_day4_final_reranker
Training minutes: 22.95
Validation Macro-F1: 0.9750


In [47]:
from pathlib import Path
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

FINAL_MODEL_DIR = Path("/kaggle/working/week3_day4_final_reranker")

if not (FINAL_MODEL_DIR / "config.json").exists():
    raise FileNotFoundError(
        f"Trained reranker checkpoint not found: {FINAL_MODEL_DIR}. "
        "Run the final Day 4 training cell before Day 6 evaluation."
    )

day6_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
day6_tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL_DIR)
day6_model = AutoModelForSequenceClassification.from_pretrained(FINAL_MODEL_DIR).to(day6_device)
day6_model.eval()

print("Week 3 - Day 6 trained checkpoint loaded")
print("Checkpoint:", FINAL_MODEL_DIR)
print("Device:", day6_device)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Week 3 - Day 6 trained checkpoint loaded
Checkpoint: /kaggle/working/week3_day4_final_reranker
Device: cuda


In [48]:
if "ranking_candidate_df" not in globals():
    raise NameError("Run Day 5 candidate generation before Day 6 evaluation.")

print("Week 3 - Day 6 evaluation setup completed")
print("Candidate dataset shape:", ranking_candidate_df.shape)
print("Candidate depth per query:", ranking_candidate_df.groupby("query_id").size().iloc[0])


Week 3 - Day 6 evaluation setup completed
Candidate dataset shape: (20000, 10)
Candidate depth per query: 20


In [49]:
DAY6_BATCH_SIZE = 16
DAY6_MAX_LENGTH = 512
reranker_scores = []

for start_index in range(0, len(ranking_candidate_df), DAY6_BATCH_SIZE):
    batch_df = ranking_candidate_df.iloc[start_index : start_index + DAY6_BATCH_SIZE]
    batch_inputs = day6_tokenizer(
        batch_df["question"].tolist(),
        batch_df["context"].tolist(),
        padding=True,
        truncation="only_second",
        max_length=DAY6_MAX_LENGTH,
        return_tensors="pt",
    )
    batch_inputs = {key: value.to(day6_device) for key, value in batch_inputs.items()}
    with torch.inference_mode():
        logits = day6_model(**batch_inputs).logits
    reranker_scores.extend(torch.softmax(logits, dim=-1)[:, 1].cpu().tolist())

ranking_candidate_df = ranking_candidate_df.copy()
ranking_candidate_df["reranker_score"] = reranker_scores

print("Cross-Encoder scoring completed")
print("Scored candidates:", len(ranking_candidate_df))
print("Score range:", f"{ranking_candidate_df['reranker_score'].min():.6f}", "to", f"{ranking_candidate_df['reranker_score'].max():.6f}")


Cross-Encoder scoring completed
Scored candidates: 20000
Score range: 0.000696 to 0.999278


In [50]:
query_ranking_records = []

for query_id, query_group in ranking_candidate_df.groupby("query_id", sort=False):
    gold_rows = query_group[query_group["label"] == 1]
    dense_rank = int(gold_rows.iloc[0]["dense_rank"]) if len(gold_rows) else None
    reranked_group = query_group.sort_values("reranker_score", ascending=False).reset_index(drop=True)
    reranked_gold_rows = reranked_group[reranked_group["label"] == 1]
    reranker_rank = int(reranked_gold_rows.index[0] + 1) if len(reranked_gold_rows) else None
    query_ranking_records.append(
        {
            "query_id": query_id,
            "dense_rank": dense_rank,
            "reranker_rank": reranker_rank,
            "gold_in_candidate_set": dense_rank is not None,
        }
    )

query_ranking_df = pd.DataFrame(query_ranking_records)
print("Query-level ranking results created")
print("Number of queries:", len(query_ranking_df))


Query-level ranking results created
Number of queries: 1000


In [51]:
# 순위 지표 계산
from math import log2

def recall_at_k(ranks, cutoff):
    return float(np.mean([rank is not None and rank <= cutoff for rank in ranks]))

def mrr_at_k(ranks, cutoff):
    return float(np.mean([1.0 / rank if rank is not None and rank <= cutoff else 0.0 for rank in ranks]))

def ndcg_at_k(ranks, cutoff):
    return float(np.mean([1.0 / log2(rank + 1) if rank is not None and rank <= cutoff else 0.0 for rank in ranks]))

dense_ranks = query_ranking_df["dense_rank"].tolist()
reranker_ranks = query_ranking_df["reranker_rank"].tolist()

day6_metrics = {"number_of_queries": len(query_ranking_df)}
for cutoff in (1, 5, 10, 20):
    day6_metrics[f"dense_recall_at_{cutoff}"] = recall_at_k(dense_ranks, cutoff)
    day6_metrics[f"reranker_recall_at_{cutoff}"] = recall_at_k(reranker_ranks, cutoff)
day6_metrics["dense_mrr_at_10"] = mrr_at_k(dense_ranks, 10)
day6_metrics["reranker_mrr_at_10"] = mrr_at_k(reranker_ranks, 10)
day6_metrics["dense_ndcg_at_10"] = ndcg_at_k(dense_ranks, 10)
day6_metrics["reranker_ndcg_at_10"] = ndcg_at_k(reranker_ranks, 10)

print("Week 3 - Day 6 Ranking Evaluation")
for metric_name, metric_value in day6_metrics.items():
    print(f"{metric_name}: {metric_value:.4f}" if isinstance(metric_value, float) else f"{metric_name}: {metric_value}")


Week 3 - Day 6 Ranking Evaluation
number_of_queries: 1000
dense_recall_at_1: 0.4610
reranker_recall_at_1: 0.7520
dense_recall_at_5: 0.6460
reranker_recall_at_5: 0.8100
dense_recall_at_10: 0.7380
reranker_recall_at_10: 0.8160
dense_recall_at_20: 0.8180
reranker_recall_at_20: 0.8180
dense_mrr_at_10: 0.5437
reranker_mrr_at_10: 0.7796
dense_ndcg_at_10: 0.5899
reranker_ndcg_at_10: 0.7887


In [52]:
positive_score_mean = ranking_candidate_df.loc[ranking_candidate_df["label"] == 1, "reranker_score"].mean()
negative_score_mean = ranking_candidate_df.loc[ranking_candidate_df["label"] == 0, "reranker_score"].mean()

print("Reranker score diagnostics")
print("Positive score mean:", f"{positive_score_mean:.6f}")
print("Negative score mean:", f"{negative_score_mean:.6f}")
print("Score difference:", f"{positive_score_mean - negative_score_mean:.6f}")


Reranker score diagnostics
Positive score mean: 0.975086
Negative score mean: 0.068502
Score difference: 0.906584


In [53]:
failure_count = 0
for query_id, query_group in ranking_candidate_df.groupby("query_id", sort=False):
    if failure_count >= 5:
        break
    reranked_group = query_group.sort_values("reranker_score", ascending=False).reset_index(drop=True)
    gold_rows = reranked_group[reranked_group["label"] == 1]
    if gold_rows.empty:
        continue
    reranker_rank = int(gold_rows.index[0] + 1)
    if reranker_rank > 1:
        gold_row = gold_rows.iloc[0]
        top_row = reranked_group.iloc[0]
        print(f"Failure case {failure_count + 1}")
        print("Question:", gold_row["question"])
        print("Gold rank:", reranker_rank)
        print("Gold score:", f"{gold_row['reranker_score']:.6f}")
        print("Top-1 score:", f"{top_row['reranker_score']:.6f}")
        print("Gold context:", gold_row["context"][:200])
        print("Top-1 context:", top_row["context"][:200])
        print("-" * 70)
        failure_count += 1

print("Displayed failure cases:", failure_count)


Failure case 1
Question: 대신을 임명하는 직책은?
Gold rank: 7
Gold score: 0.999098
Top-1 score: 0.999244
Gold context: 의회에 향한 책임정부는 여러가지 방법으로 나타난다. 대신들은 자기들의 결정과 부처의 실적을 의회의 의원들에게 보고한다. 의회 안에서 발표하는 일과 질의응답을 수행하는 요건은 대신들도 의원들이 수행하는 권리를 반드시 있여야 한다. 또한 국가 원수격인 군주 또는 총독이 대신을 공식적으로 임명한다. 그러나 이론상 군주의 분부대로 대신을 해고할 수 있지만 그들은 
Top-1 context: 1882년 구미 열강의 통상 요구와 임오군란의 뒤처리 등 복잡한 문제에 부딪친 조선 정부에 다시 기용되었다. 전권대신 이유원(李裕元)의 부관으로서 일본과 제물포 조약을 맺는 데 앞장서는 등 여러 나라와 수호조약(한미(韓美)·한영(韓英)·한독(韓獨) 등) 등을 맺을 때 유효적절한 수완을 발휘하여 ‘비오는 날의 나막신’에 비유되곤 하였다. 한학에 능통했던 그는
----------------------------------------------------------------------
Failure case 2
Question: 러시아가 퇴각을 멈추고 본대와 합류 후 프랑스군과 전투를 치른 전투는 무엇인가?
Gold rank: 2
Gold score: 0.998088
Top-1 score: 0.998471
Gold context: 그러나 우세를 이용해 러시아군 본대와 승부를 가리기로 마음먹은 나폴레옹은 추격을 곧 재개했고 8월 초에 드디어 합류하는 데 성공한 러시아군도 퇴각을 그만두고 전투 태세에 들어가 8월 중반 스몰렌스크 전투가 벌어지게 되었다. 데 톨리는 스몰렌스크를 중심으로 방어전을 전개했는데 바그라티온이 이끌고 온 3만여 명은 도시를 방위하도록 하고 나머지 10만여 병력은 
Top-1 context: 협정이 성사되지 않은데다가 코자크 기병대가 수송대와 전령을 습격하는 일이 잦아지고 겨

# Day 6 Experiment Log

Reranking improved Recall@1 from 0.4610 to 0.7520, MRR@10 from 0.5437 to 0.7796, and nDCG@10 from 0.5899 to 0.7887. Recall@20 remained 0.8180 because the candidate set did not change.


# Day 7 - Dense Retriever and Reranker Pipeline Modularization

Verify the reusable dense-retriever and reranker pipeline.


In [54]:
required_day7_variables = [
    "week2_encoder",
    "retrieval_index",
    "corpus_df",
    "day6_model",
    "day6_tokenizer",
]

missing_day7_variables = [
    variable_name
    for variable_name in required_day7_variables
    if variable_name not in globals()
]

if missing_day7_variables:
    raise NameError(
        "Missing required variables: "
        f"{missing_day7_variables}"
    )

day7_device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Week 3 - Day 7 setup completed")
print("Device:", day7_device)
print("Corpus size:", len(corpus_df))
print("Embedding dimension:", retrieval_index.d)

Week 3 - Day 7 setup completed
Device: cuda
Corpus size: 960
Embedding dimension: 768


In [55]:
class DenseRetriever:
    def __init__(
        self,
        encoder,
        index,
        corpus_dataframe
    ):
        self.encoder = encoder
        self.index = index
        self.corpus_dataframe = (
            corpus_dataframe.reset_index(drop=True)
        )

    def retrieve(
        self,
        queries,
        top_k=10
    ):
        query_embeddings = self.encoder.encode(
            queries
        )

        query_embeddings = np.asarray(
            query_embeddings,
            dtype="float32"
        )

        faiss.normalize_L2(
            query_embeddings
        )

        similarities, indices = (
            self.index.search(
                query_embeddings,
                top_k
            )
        )

        retrieval_results = []

        for query_index, query in enumerate(queries):
            query_results = []

            for rank, (
                corpus_index,
                similarity
            ) in enumerate(
                zip(
                    indices[query_index],
                    similarities[query_index]
                ),
                start=1
            ):
                corpus_index = int(corpus_index)

                corpus_row = (
                    self.corpus_dataframe
                    .iloc[corpus_index]
                )

                query_results.append(
                    {
                        "query": query,
                        "rank": rank,
                        "corpus_index": corpus_index,
                        "title": corpus_row["title"],
                        "context": corpus_row["context"],
                        "dense_similarity": float(
                            similarity
                        )
                    }
                )

            retrieval_results.extend(
                query_results
            )

        return pd.DataFrame(
            retrieval_results
        )

In [56]:
class CrossEncoderReranker:
    def __init__(
        self,
        model,
        tokenizer,
        device,
        batch_size=32,
        max_length=512
    ):
        self.model = model.to(device)
        self.tokenizer = tokenizer
        self.device = device
        self.batch_size = batch_size
        self.max_length = max_length

        self.model.eval()

    def score(
        self,
        questions,
        contexts
    ):
        scores = []

        for start_index in range(
            0,
            len(questions),
            self.batch_size
        ):
            batch_questions = questions[
                start_index:
                start_index + self.batch_size
            ]

            batch_contexts = contexts[
                start_index:
                start_index + self.batch_size
            ]

            batch_inputs = self.tokenizer(
                batch_questions,
                batch_contexts,
                padding=True,
                truncation="only_second",
                max_length=self.max_length,
                return_tensors="pt"
            )

            batch_inputs = {
                key: value.to(self.device)
                for key, value in batch_inputs.items()
            }

            with torch.inference_mode():
                batch_outputs = self.model(
                    **batch_inputs
                )

            relevance_scores = torch.softmax(
                batch_outputs.logits,
                dim=-1
            )[:, 1]

            scores.extend(
                relevance_scores
                .cpu()
                .numpy()
                .tolist()
            )

        return np.asarray(
            scores,
            dtype="float32"
        )

    def rerank(
        self,
        retrieval_dataframe
    ):
        reranked_dataframe = (
            retrieval_dataframe.copy()
        )

        reranked_dataframe[
            "reranker_score"
        ] = self.score(
            reranked_dataframe["query"].tolist(),
            reranked_dataframe["context"].tolist()
        )

        reranked_dataframe = (
            reranked_dataframe
            .sort_values(
                by=[
                    "query",
                    "reranker_score"
                ],
                ascending=[
                    True,
                    False
                ]
            )
            .copy()
        )

        reranked_dataframe[
            "reranker_rank"
        ] = (
            reranked_dataframe
            .groupby("query")
            .cumcount()
            + 1
        )

        return reranked_dataframe

In [57]:
day7_dense_retriever = DenseRetriever(
    encoder=week2_encoder,
    index=retrieval_index,
    corpus_dataframe=corpus_df
)

day7_reranker = CrossEncoderReranker(
    model=day6_model,
    tokenizer=day6_tokenizer,
    device=day7_device,
    batch_size=32,
    max_length=512
)

test_queries = [
    "이명박은 언제 취임식을 가졌는가?",
    "2006년 강정호는 어디의 1순위 지명을 받아 입단하였는가?"
]

day7_dense_results = (
    day7_dense_retriever.retrieve(
        test_queries,
        top_k=10
    )
)

day7_reranked_results = (
    day7_reranker.rerank(
        day7_dense_results
    )
)

print("Day 7 modular pipeline test completed")
print(
    "Dense retrieval rows:",
    len(day7_dense_results)
)
print(
    "Reranked rows:",
    len(day7_reranked_results)
)

display(
    day7_reranked_results[
        [
            "query",
            "rank",
            "reranker_rank",
            "dense_similarity",
            "reranker_score",
            "context"
        ]
    ]
)


Day 7 modular pipeline test completed
Dense retrieval rows: 20
Reranked rows: 20


,query,rank,reranker_rank,dense_similarity,reranker_score,context
10,2006년 강정호는 어디의 1순위 지명을 받아 입단하였는가?,1,1,0.615990,0.999234,2006년 현대 유니콘스의 2차 1순위 지명을 받아 입단하였다. 지명 당시 포지션은...
12,2006년 강정호는 어디의 1순위 지명을 받아 입단하였는가?,3,2,0.466248,0.000857,2010년 아시안 게임에서의 대활약으로 시즌 전부터 엄청난 주목을 받았고 메이저 리...
18,2006년 강정호는 어디의 1순위 지명을 받아 입단하였는가?,9,3,0.440152,0.000851,Rickenbacker(리켄베커) 기타의 탄생이 전기 기타의 최초라고 정의한다. 전...
13,2006년 강정호는 어디의 1순위 지명을 받아 입단하였는가?,4,4,0.459003,0.000825,"충주중학교를 졸업하였고, 충주고등학교 2학년 때 미국 적십자사에서 주최하는 영어경시..."
17,2006년 강정호는 어디의 1순위 지명을 받아 입단하였는가?,8,5,0.441114,0.000815,2006년 2월 14일에 유엔 차기 사무총장 선거에 공식적으로 출마 선언을 하여 선...
19,2006년 강정호는 어디의 1순위 지명을 받아 입단하였는가?,10,6,0.438918,0.000793,3월 10일 현대 유니콘스가 해체되고 우리 히어로즈로 선수단이 인계된 이후 1군에서...
15,2006년 강정호는 어디의 1순위 지명을 받아 입단하였는가?,6,7,0.455878,0.000791,"독일에서 영구 귀국한 후 ""차범근 축구교실""을 운영하기 시작하였다. 1990년 11..."
14,2006년 강정호는 어디의 1순위 지명을 받아 입단하였는가?,5,8,0.458742,0.000772,2003년 '유아인'이라는 예명을 사용하면서 라면 광고로 데뷔했다. 같은 해 10월...
16,2006년 강정호는 어디의 1순위 지명을 받아 입단하였는가?,7,9,0.453598,0.000763,"2007년, Pussycat Dolls Present: The Search For ..."
11,2006년 강정호는 어디의 1순위 지명을 받아 입단하였는가?,2,10,0.524675,0.000751,2015년 1월에 메이저리그의 피츠버그 파이리츠와 4년간 보장금액 1100만달러에 ...


In [58]:
if "day7_reranked_results" not in globals():
    raise NameError(
        "day7_reranked_results is not available. "
        "Run the Day 7 modular pipeline test first."
    )

day7_analysis_df = (
    day7_reranked_results
    .copy()
    .reset_index(drop=True)
)

print("Day 7 analysis dataframe created")
print(
    "Number of queries:",
    day7_analysis_df["query"].nunique()
)
print(
    "Number of candidates:",
    len(day7_analysis_df)
)

Day 7 analysis dataframe created
Number of queries: 2
Number of candidates: 20


In [59]:
day7_analysis_path = (
    "/kaggle/working/"
    "week3_day7_pipeline_test_results.csv"
)

day7_analysis_df.to_csv(
    day7_analysis_path,
    index=False,
    encoding="utf-8-sig"
)

print("Day 7 analysis results saved to:")
print(day7_analysis_path)

Day 7 analysis results saved to:
/kaggle/working/week3_day7_pipeline_test_results.csv


# Day 7 Experiment Log

Verified that the saved retriever and reranker work together outside training.
